# Digital Clone

Turn on GPU: **Settings → Accelerator → GPU T4 x2** (or P100).

Two input modes, set by `MODE` below:
- `"video"` — animates `face_clip.mp4` (from a recorded clip) with Wav2Lip.
- `"image"` — animates a single still `photo.jpg` with SadTalker. Works from
  just one picture, and generates video for however long the script runs
  (not capped to a short loop).

Attach a private dataset with `voice_reference.wav` plus either
`face_clip.mp4` or `photo.jpg` under **Add Input**, edit `SCRIPT_TEXT` and
`MODE` in the next cell, then **Run All**. Output lands at
`/kaggle/working/output.mp4`.

In [ ]:
# --- Parameters ---
SCRIPT_TEXT = (
    "Hi, this is my digital clone. Everything you hear was typed, not recorded."
)

MODE = "video"  # "video" (Wav2Lip, needs face_clip.mp4) or "image" (SadTalker, needs photo.jpg)

DATASET_DIR = "/kaggle/input/digital-clone-assets"  # rename to match your dataset
VOICE_REFERENCE = f"{DATASET_DIR}/voice_reference.wav"
FACE_CLIP = f"{DATASET_DIR}/face_clip.mp4"
IMAGE_PATH = f"{DATASET_DIR}/photo.jpg"

WORK_DIR = "/kaggle/working"
GENERATED_SPEECH = f"{WORK_DIR}/generated_speech.wav"
OUTPUT_VIDEO = f"{WORK_DIR}/output.mp4"

## 1. Install dependencies

In [ ]:
!pip install -q TTS
!git clone -q https://github.com/Rudrabha/Wav2Lip.git /kaggle/working/Wav2Lip
!pip install -q -r /kaggle/working/Wav2Lip/requirements.txt || true
!git clone -q https://github.com/OpenTalker/SadTalker.git /kaggle/working/SadTalker
!pip install -q -r /kaggle/working/SadTalker/requirements.txt || true

import os
os.makedirs("/kaggle/working/Wav2Lip/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/Wav2Lip/face_detection/detection/sfd", exist_ok=True)
os.makedirs("/kaggle/working/SadTalker/checkpoints", exist_ok=True)

## 2. Fetch Wav2Lip checkpoints

The upstream Google Drive links are frequently dead. If the download below
fails, attach a Kaggle dataset with `wav2lip_gan.pth` and `s3fd.pth` as a
second input and point `CKPT_SRC` / `S3FD_SRC` at it instead.

In [ ]:
import shutil, urllib.request

CKPT_SRC = None  # e.g. "/kaggle/input/wav2lip-checkpoints/wav2lip_gan.pth"
S3FD_SRC = None  # e.g. "/kaggle/input/wav2lip-checkpoints/s3fd.pth"

CKPT_DST = "/kaggle/working/Wav2Lip/checkpoints/wav2lip_gan.pth"
S3FD_DST = "/kaggle/working/Wav2Lip/face_detection/detection/sfd/s3fd.pth"

CKPT_URL = "https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth"
S3FD_URL = "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth"

if CKPT_SRC:
    shutil.copy(CKPT_SRC, CKPT_DST)
else:
    urllib.request.urlretrieve(CKPT_URL, CKPT_DST)

if S3FD_SRC:
    shutil.copy(S3FD_SRC, S3FD_DST)
else:
    urllib.request.urlretrieve(S3FD_URL, S3FD_DST)

print("Checkpoints ready.")

## 2b. Fetch SadTalker checkpoints (only needed for `MODE = "image"`)

If the download script's mirrors are down, attach a Kaggle dataset with the
SadTalker `checkpoints/` and `gfpgan/weights/` folders and point
`SADTALKER_CKPT_SRC` at it instead.

In [ ]:
import re
import numpy as np
import soundfile as sf
from TTS.api import TTS

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")


def synthesize_speech(text: str, out_path: str, pause_ms: int = 250) -> None:
    """Clone-voice TTS for scripts of any length (XTTS chokes on very long
    single calls, so split on sentences and stitch the audio back together)."""
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]
    rate = tts.synthesizer.output_sample_rate
    silence = np.zeros(int(rate * pause_ms / 1000), dtype=np.float32)

    chunks = []
    for sentence in sentences:
        wav = tts.tts(text=sentence, speaker_wav=VOICE_REFERENCE, language="en")
        chunks.append(np.asarray(wav, dtype=np.float32))
        chunks.append(silence)

    audio = np.concatenate(chunks) if chunks else silence
    sf.write(out_path, audio, rate)


synthesize_speech(SCRIPT_TEXT, GENERATED_SPEECH)
print("Generated:", GENERATED_SPEECH)

## 4. Animate: Wav2Lip (video mode) or SadTalker (image mode)

In [ ]:
import glob
import shutil
import subprocess


def generate_avatar_clip(audio_path: str, out_path: str) -> None:
    if MODE == "image":
        result_dir = f"{WORK_DIR}/sadtalker_tmp"
        shutil.rmtree(result_dir, ignore_errors=True)
        os.makedirs(result_dir, exist_ok=True)
        subprocess.run(
            [
                "python", "inference.py",
                "--driven_audio", audio_path,
                "--source_image", IMAGE_PATH,
                "--result_dir", result_dir,
                "--still", "--preprocess", "full", "--enhancer", "gfpgan",
            ],
            cwd=SADTALKER_DIR, check=True,
        )
        produced = sorted(
            glob.glob(f"{result_dir}/**/*.mp4", recursive=True), key=os.path.getmtime
        )
        if not produced:
            raise RuntimeError("SadTalker did not produce an output video")
        shutil.copy(produced[-1], out_path)
    else:
        subprocess.run(
            [
                "python", "inference.py",
                "--checkpoint_path", "checkpoints/wav2lip_gan.pth",
                "--face", FACE_CLIP,
                "--audio", audio_path,
                "--outfile", out_path,
            ],
            cwd="/kaggle/working/Wav2Lip", check=True,
        )


generate_avatar_clip(GENERATED_SPEECH, OUTPUT_VIDEO)
print("Wrote:", OUTPUT_VIDEO)

## 5. Preview

In [ ]:
from IPython.display import Video
Video(OUTPUT_VIDEO, embed=True, width=480)

## 6. Multi-slide narration (optional — for a PPT demo video)

Skip this section for a single clip. For a full slide-by-slide demo:

1. Locally, run `scripts/ppt_to_slides.py` on your deck to get
   `slide_001.png`, `slide_002.png`, …
2. Write `slides_script.json` — one narration segment per slide, in order:
   ```json
   [
     {"slide": "slide_001.png", "text": "..."},
     {"slide": "slide_002.png", "text": "..."}
   ]
   ```
3. Add `slides_script.json` to the same Kaggle dataset as your voice/face
   assets, then run the cell below. It writes one avatar clip per slide to
   `/kaggle/working/segments/avatar_XXX.mp4` — same voice, same face clip,
   just a different line each time.
4. Download the `segments/` folder. Locally (CPU-only, no Kaggle needed),
   composite each avatar clip onto its slide with
   `scripts/compose_slide_avatar.py`, then stitch them into the final demo
   with `scripts/concat_segments.py`.

In [ ]:
import json

SLIDES_SCRIPT_PATH = f"{DATASET_DIR}/slides_script.json"
SEGMENTS_DIR = f"{WORK_DIR}/segments"
os.makedirs(SEGMENTS_DIR, exist_ok=True)

if os.path.exists(SLIDES_SCRIPT_PATH):
    with open(SLIDES_SCRIPT_PATH) as f:
        slides = json.load(f)

    for i, item in enumerate(slides, start=1):
        speech_path = f"{SEGMENTS_DIR}/speech_{i:03d}.wav"
        avatar_path = f"{SEGMENTS_DIR}/avatar_{i:03d}.mp4"

        synthesize_speech(item["text"], speech_path)
        generate_avatar_clip(speech_path, avatar_path)

        print(f"Slide {i}: {avatar_path}")

    print(f"\nDone — {len(slides)} avatar clip(s) in {SEGMENTS_DIR}/")
else:
    print(f"No {SLIDES_SCRIPT_PATH} found — skipping multi-slide mode.")